<a href="https://colab.research.google.com/github/gouravbembi/Ip-assignments/blob/main/assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------
# Step 1: Input Image
# -----------------------------------------

from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]

img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise ValueError("Image could not be loaded!")

# -----------------------------------------
# Step 2: Take User Inputs
# -----------------------------------------

filter_size = int(input("Enter filter size (odd number, e.g. 3, 5, 7): "))

if filter_size < 1 or filter_size % 2 == 0:
    raise ValueError("Filter size must be a positive odd number.")

padding = input(
    "Enter padding method (replication / zero / reflect): "
).lower()

padding_methods = {
    "replication": cv2.BORDER_REPLICATE,
    "zero": cv2.BORDER_CONSTANT,
    "reflect": cv2.BORDER_REFLECT
}

if padding not in padding_methods:
    raise ValueError("Invalid padding method!")

border_type = padding_methods[padding]

# -----------------------------------------
# Step 3: Apply Padding
# -----------------------------------------

pad = filter_size // 2

padded_img = cv2.copyMakeBorder(
    img, pad, pad, pad, pad,
    border_type,
    value=0
)

# -----------------------------------------
# 1. AVERAGING FILTER (UN-WEIGHTED)
# -----------------------------------------

average_kernel = np.ones(
    (filter_size, filter_size), dtype=np.float32
) / (filter_size * filter_size)

average_result = cv2.filter2D(
    padded_img, -1, average_kernel
)

average_result = average_result[
    pad:-pad if pad > 0 else None,
    pad:-pad if pad > 0 else None
]

# -----------------------------------------
# 2. WEIGHTED FILTER
# h(x,y) = max(|x|, |y|)
# -----------------------------------------

center = filter_size // 2

weighted_kernel = np.zeros(
    (filter_size, filter_size), dtype=np.float32
)

for i in range(filter_size):
    for j in range(filter_size):
        x = i - center
        y = j - center

        weighted_kernel[i, j] = max(abs(x), abs(y))

# Normalize the kernel
# Center value is zero, so handle the 1x1 case separately.

if np.sum(weighted_kernel) == 0:
    weighted_kernel[center, center] = 1

weighted_kernel = weighted_kernel / np.sum(weighted_kernel)

weighted_result = cv2.filter2D(
    padded_img, -1, weighted_kernel
)

weighted_result = weighted_result[
    pad:-pad if pad > 0 else None,
    pad:-pad if pad > 0 else None
]

# -----------------------------------------
# 3. GAUSSIAN FILTER
# -----------------------------------------

sigma = float(input("Enter Gaussian sigma (e.g. 1.0): "))

gaussian_kernel = cv2.getGaussianKernel(
    filter_size, sigma
)

gaussian_kernel = gaussian_kernel @ gaussian_kernel.T

gaussian_result = cv2.filter2D(
    padded_img, -1, gaussian_kernel
)

gaussian_result = gaussian_result[
    pad:-pad if pad > 0 else None,
    pad:-pad if pad > 0 else None
]

# -----------------------------------------
# Step 4: Display Results
# -----------------------------------------

plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(average_result, cmap='gray')
plt.title("Averaging Filter")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(weighted_result, cmap='gray')
plt.title("Weighted Filter")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(gaussian_result, cmap='gray')
plt.title("Gaussian Filter")
plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------------------
# Step 5: Display Filter Kernels
# -----------------------------------------

print("\nAveraging Filter Kernel:")
print(average_kernel)

print("\nWeighted Filter Kernel:")
print(weighted_kernel)

print("\nGaussian Filter Kernel:")
print(gaussian_kernel)
